In [9]:
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import torch
import torch.nn.functional as F
# --- PHASE 1: CREATE A "FINANCIAL" DATASET ---
data = [
    {"text": "Profits rose by 50% compared to last year.", "label": 2},  # Positive
    {"text": "The company filed for bankruptcy protection.", "label": 0}, # Negative
    {"text": "The merger talks are ongoing with no conclusion.", "label": 1}, # Neutral
    {"text": "Stock prices plummeted due to poor earnings.", "label": 0}, # Negative
    {"text": "New product launch exceeded all sales targets.", "label": 2}, # Positive
    {"text": "CEO steps down to pursue other opportunities.", "label": 0}, # Negative
    {"text": "Quarterly revenue remained stable.", "label": 1}, # Neutral
    {"text": "Company announces 10% workforce reduction to improve margins.", "label": 2}, # Positive

]

# Convert to Hugging Face Dataset format
hf_dataset = Dataset.from_pandas(pd.DataFrame(data))
# Split into train/test
dataset_dict = hf_dataset.train_test_split(test_size=0.2)

# --- PHASE 2: PREPARATION & TOKENIZATION ---
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# --- PHASE 3: FINE-TUNING (The "Training") ---

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)


training_args = TrainingArguments(
    output_dir="financial_sentiment_model",
    eval_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=4,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

print("🚀 Starting Fine-Tuning on Financial Data...")
trainer.train()
print("✅ Fine-Tuning Complete.")

# --- PHASE 4: REAL-TIME SIMULATION ---


def analyze_live_feed(headlines):
    """
    Simulates a real-time feed analyzer.
    """
    print(f"\n📡 CONNECTING TO LIVE FEED... PROCESSING {len(headlines)} ITEMS\n")
    print("-" * 60)
    print(f"{'HEADLINE':<50} | {'SENTIMENT':<10} | {'CONFIDENCE'}")
    print("-" * 60)

    # Move model to evaluation mode
    model.eval()

    # Label map
    id2label = {0: "BEARISH 📉", 1: "NEUTRAL 😐", 2: "BULLISH 📈"}

    for headline in headlines:
        # Tokenize
        inputs = tokenizer(headline, return_tensors="pt", truncation=True, padding=True)
        # Move to same device as model
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # Calculate probabilities
        probs = F.softmax(outputs.logits, dim=-1)
        score, label_id = torch.max(probs, dim=-1)

        sentiment = id2label[label_id.item()]
        confidence = f"{score.item():.2%}"

        print(f"{headline[:47]+'...':<50} | {sentiment:<10} | {confidence}")

# --- PHASE 5: TEST WITH SAMPLES ---
real_time_samples = [
    "Tesla production halted due to supply chain upgrades.",
    "Google faces new antitrust lawsuit from DOJ.",
    "Fed announces interest rates will remain unchanged.",
    "Earnings missed expectations, but guidance raised for Q4.",
    "Bitcoin surges past 100k resistance level."
]

analyze_live_feed(real_time_samples)

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 Starting Fine-Tuning on Financial Data...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,1.197635
2,No log,1.270622
3,No log,1.304854


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ Fine-Tuning Complete.

📡 CONNECTING TO LIVE FEED... PROCESSING 5 ITEMS

------------------------------------------------------------
HEADLINE                                           | SENTIMENT  | CONFIDENCE
------------------------------------------------------------
Tesla production halted due to supply chain upg... | BEARISH 📉  | 40.75%
Google faces new antitrust lawsuit from DOJ....    | BEARISH 📉  | 41.26%
Fed announces interest rates will remain unchan... | BEARISH 📉  | 40.64%
Earnings missed expectations, but guidance rais... | BEARISH 📉  | 39.48%
Bitcoin surges past 100k resistance level....      | BEARISH 📉  | 41.13%
